# Lab 0 · Python and data preparation
**Course:** AI in Oil & Gas · Part I, AI & ML Fundamentals  ·  **Time:** 60 min  ·  **Slides:** Modules 04 and 08

Before any algorithm, look at the data. In this lab you load the three course datasets, find what is wrong with
them, and prepare them so a model can learn.

| Dataset | File | Task | Rows | Inputs → target |
|---|---|---|---|---|
| Asphaltene precipitation | `asphaltene.csv` | regression | 167 | temperature, n-alkane carbon number, dilution ratio → wt% precipitated |
| EOR screening | `eor_screening.csv` | classification | 455 | permeability, depth, API, viscosity, temperature, oil saturation → EOR method |
| Virtual flow metering | `vfm_well_rates.csv` | regression | 2,100 | choke, BHP, WHP, WHT, Psep, Tsep → oil, water, gas rates |

**You will:** load and describe data · plot distributions · detect outliers with the IQR rule · transform skewed inputs · scale · split.

In [ ]:
# --- setup: works locally and in Google Colab ---
import os, sys
if "google.colab" in sys.modules and not os.path.exists("aiml"):
    print("Upload the course kit zip to Colab, then unzip it:  !unzip -q aiml_course_kit.zip && %cd aiml_course_kit")
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})
DATA = "../data" if os.path.exists("../data") else "data"

from aiml.data import load_asphaltene, load_eor, load_vfm, ASPH_FEATURES, ASPH_TARGET, EOR_FEATURES
from aiml.prep import detect_outliers_iqr, outlier_table, plot_outliers
asph = pd.read_csv(f"{DATA}/asphaltene.csv")
eor = pd.read_csv(f"{DATA}/eor_screening.csv")
vfm = pd.read_csv(f"{DATA}/vfm_well_rates.csv")
asph.head()

## 1 · First look
`info()` shows types and missing values; `describe()` the ranges. Read them like an engineer: are the units and ranges physical?

In [ ]:
asph.info(); asph.describe().T

In [ ]:
eor.describe().T.round(1)

### ✏️ Your turn 1
In `eor`, find (a) how many projects report a viscosity of **0 cP**, (b) the lowest reservoir temperature, and
(c) how many rows are exact duplicates in the six inputs and the label. Which of these are data errors?

In [ ]:
# (a) viscosity == 0
# (b) minimum temperature
# (c) duplicated rows (inputs + label)

## 2 · Distributions
Skewed inputs (viscosity, permeability) span several orders of magnitude: a model sees almost all points squeezed near zero.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
eor.viscosity_cp.plot.hist(bins=40, ax=ax[0], title="viscosity (cP)")
np.log10(eor.viscosity_cp.clip(lower=0.1)).plot.hist(bins=40, ax=ax[1], title="log10 viscosity")
eor.eor_method.value_counts().plot.bar(ax=ax[2], title="EOR method (class balance)", rot=30)
plt.tight_layout(); plt.show()

## 3 · Outliers with the IQR rule
The course function `detect_outliers_iqr` flags values outside Q1 − 1.5·IQR and Q3 + 1.5·IQR.

In [ ]:
outlier_table(asph)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
plot_outliers(asph, ASPH_TARGET, ax=ax[0])
for (T, C), g in asph.groupby(["temperature_K", "alkane_carbon_number"]):
    ax[1].plot(g.dilution_ratio, g[ASPH_TARGET], "o-", ms=3, alpha=.8, color="tab:red" if T == 332 and C <= 8 else "tab:grey")
ax[1].set_xlabel("dilution ratio (mL/g)"); ax[1].set_ylabel("wt% precipitated"); ax[1].set_title("each line = one temperature + n-alkane series")
plt.tight_layout(); plt.show()

The IQR rule flags 15 values. Plotted by series (right), they are not random spikes: they belong to **three complete
series** of 17 tests (332 K with n-C6, C7, C8) that are about ten times higher than every other series, while 332 K with C9–C12 looks normal. A statistical rule finds
them; only engineering judgement can decide whether they are a different unit, a different oil or a real effect.

`load_asphaltene()` keeps them and adds a `suspect` column so you can model with and without them (Lab 1).

### ✏️ Your turn 2
Run `detect_outliers_iqr` on every numeric column of `vfm`. Are any of the flagged values *errors*? (Hint: look at the
water rates and remember that water cut goes from 0 to 0.8 in the tests.)

In [ ]:
# loop over the numeric columns of vfm and print how many outliers each has

## 4 · A hidden problem: duplicated tests
The VFM file looks large (2,100 rows). How many *different* well tests does it hold?

In [ ]:
phys = ["choke_in", "bhp_psi", "whp_psi", "wht_F", "tsep_F", "oil_rate_stb_d", "water_rate_bbl_d", "gas_rate_mmscf_d"]
print("rows:", len(vfm), " different physical tests:", len(vfm[phys].drop_duplicates()))
vfm[(vfm.bhp_psi == 5410.33)][["choke_in", "bhp_psi", "whp_psi", "psep_psi", "oil_rate_stb_d"]]

The same well test is repeated for each separator pressure (100–600 psi), and the rates do not change. With a random
train/test split the "test" rows are copies of training rows, so any model looks perfect. `load_vfm()` adds a `case_id`;
always split by it (Lab 3).

## 5 · Scaling and splitting
Fit the scaler on the **training** rows only, then apply it to the test rows.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
X, y = asph[ASPH_FEATURES], asph[ASPH_TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler().fit(X_train)
print(pd.DataFrame(sc.transform(X_train), columns=ASPH_FEATURES).describe().loc[["mean", "std"]].round(3))
mm = MinMaxScaler().fit(X_train); print(pd.DataFrame(mm.transform(X_test), columns=ASPH_FEATURES).describe().loc[["min", "max"]].round(3))

### ✏️ Your turn 3
For the EOR data, make a **stratified** 80/20 split (`stratify=y`) and print the class counts of the test set.
Why is stratification important here? (Hint: how many Nitrogen projects are there?)

In [ ]:
y = eor.eor_method
# X_train, X_test, y_train, y_test = train_test_split(...)

## Key ideas
* Read `describe()` like an engineer: a 0 cP oil or a 5 °F reservoir is a data error, not a feature.
* The IQR rule finds unusual values; deciding whether they are wrong needs domain knowledge.
* Duplicated rows make random splits leak: split by the physical case.
* Log-transform inputs that span orders of magnitude; scale using training data only; stratify small classes.